In [16]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import torch
from functools import partial

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")


ROOT = Path(r"E:\positron\NFlow")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import Python.config as cfg
import Python.framework as fw
import Python.model as md

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

DATA_ID = "n160p100"
SETTING = "weak_signal"
CONFIG_NAME = "last_default"
SEED = 400

manifest_tag = "" if SETTING == "simple" else f"_{SETTING}"
manifest = pd.read_csv(ROOT / "data" / DATA_ID / f"manifest_{DATA_ID}{manifest_tag}.csv")

job = manifest[
    (manifest["setting"].astype(str) == SETTING)
    & (manifest["seed"].astype(int) == SEED)
].iloc[0]

dat = pd.read_csv(ROOT / job["data_path"])
beta_tbl = pd.read_csv(ROOT / job["beta_path"])

X = torch.tensor(dat.drop(columns=["y"]).to_numpy(np.float32), dtype=dtype, device=device)
y = torch.tensor(dat["y"].to_numpy(np.float32), dtype=dtype, device=device)
beta_true = torch.tensor(beta_tbl["beta_true"].to_numpy(np.float32), dtype=dtype, device=device)

sim_info = job.to_dict()
sim_info["data_id"] = DATA_ID
sim_info["config_name"] = CONFIG_NAME

split_cfg = cfg.SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)

In [23]:
schedule_cfg_test = cfg.StagewiseAnnealConfig(
    tau_start=0.8,
    tau_end=0.8,
    warmup_epochs=100,
    n_anneal_stages=1,
    min_stage_epochs=700,
    max_stage_epochs=700,
    base_lr=1e-5,
    stage_lr_decay=1.0,
    min_lr_scale=1.0,
    eval_every=25,
    print_every=100,
    diag_R_train=64,
    diag_R_final=4000,
    support_threshold=0.00145,
    recovery_min_epoch=25,
    recovery_score_col="moment_recovery_score",
)

In [24]:
build_vi = md.build_flow_vi

out_test = fw.simflow_stagewise(
    X=X,
    y=y,
    beta_true=beta_true,
    sim_info=sim_info,
    build_flow_vi=build_vi,
    family="gaussian",
    seed=SEED,
    K_q=64,
    K_g=4,
    schedule_cfg=schedule_cfg_test,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
    compare_mcmc=True,
    mcmc_root=str(ROOT / "data" / DATA_ID / f"{DATA_ID}_mcmc_output"),
    mcmc_setting=SETTING,
    mcmc_seed=SEED,
    mcmc_during_training=False,
)

===== Simulation info =====
Simulation summary:
  setting               : weak_signal
  seed                  : 400
  n                     : 160
  p                     : 100
  n_active              : 10
  sigma2                : 1.0000
  sigma                 : 1.0000
  signal_var            : 1.3170
  outcome_var           : 2.0646
  snr_actual            : 1.3170
  beta_low              : 0.1000
  beta_high             : 0.5000

Active indices, zero-based:
  13;19;26;32;39;50;53;82;90;98
[stage 00 | epoch 0001] tau=0.8000 lr=1.00e-05 loss=608.197021 score=5.2196 S=100 pip=51.73 churn=0.000 ok*
[stage 00 | epoch 0100] tau=0.8000 lr=1.00e-05 loss=468.994843 score=3.3867 S= 99 pip=45.48 churn=0.010 ok*
[stage 01 | epoch 0101] tau=0.8000 lr=1.00e-05 loss=467.100311 score=3.5766 S= 93 pip=41.11 churn=0.080 ok*
[stage 01 | epoch 0200] tau=0.8000 lr=1.00e-05 loss=340.274536 score=2.2579 S=  5 pip=0.20 churn=0.750 ok*
[stage 01 | epoch 0300] tau=0.8000 lr=1.00e-05 loss=203.616760 score=1.3